In [3]:
#orders
import pandas as pd
import numpy as np
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load
orders = pd.read_csv('KartZone_Orders_V2.csv')

# First look
print(orders.shape)
print(orders.head())
print(orders.info())
print(orders.describe())
print(orders.columns.tolist())

(10170, 21)
        Order_ID Customer_ID Product_ID  Order_Date Expected_Delivery_Date  \
0  ORD-KZ-100001       C1181  BEAU-4051  23.10.2024             2024-10-26   
1  ORD-KZ-100002       C1489  FASH-2059  13-08-2023             16.08.2023   
2  ORD-KZ-100003       C1772  BEAU-4103  01-05-2023             2023-05-08   
3  ORD-KZ-100004       C1287  BEAU-4048  11/07/2024             11/10/2024   
4  ORD-KZ-100005       C1516  FASH-2074  2024-12-09                    NaN   

        Category       City      Payment_Mode  Quantity      MRP  ...  \
0         Beauty  Bangalore                CC       2.0   686.82  ...   
1        FASHION    Chennai  Cash on Delivery       2.0   996.89  ...   
2         BEAUTY  BANGALORE               Upi       2.0   482.23  ...   
3  Personal Care   chennai        CREDIT CARD       1.0  1474.47  ...   
4        fashion  Bangalore       credit card       5.0  6554.65  ...   

  Selling_Price  Final_Amount Unit_Cost  Total_Cost    Profit  \
0        480.77

In [4]:
# ============================================================
# STEP 2 — STRUCTURAL CLEANING
# ============================================================

# 1. Check and remove completely blank rows
blank_rows = orders.isnull().all(axis=1).sum()
print("Completely blank rows:", blank_rows)

orders = orders.dropna(how='all').reset_index(drop=True)

print("Rows after removing blank rows:", len(orders))
print("Completely blank rows remaining:",
      orders.isnull().all(axis=1).sum())


# 2. Check and remove exact duplicate rows
exact_duplicates = orders.duplicated().sum()
print("Exact duplicate rows:", exact_duplicates)

print("Before removing exact duplicates:", len(orders))

orders = orders.drop_duplicates().reset_index(drop=True)

print("After removing exact duplicates:", len(orders))
print("Exact duplicates remaining:", orders.duplicated().sum())


# 3. Validate Order_ID uniqueness
duplicate_order_ids = orders['Order_ID'].duplicated().sum()

print("Duplicate Order_IDs:", duplicate_order_ids)

if duplicate_order_ids > 0:
    print("\nDuplicate Order_ID examples:")
    print(
        orders[orders['Order_ID'].duplicated(keep=False)]
        .sort_values('Order_ID')
        .head(10)
    )


# 4. Category inspection
print("\nCategory distribution:")
print(orders['Category'].value_counts(dropna=False))

Completely blank rows: 20
Rows after removing blank rows: 10150
Completely blank rows remaining: 0
Exact duplicate rows: 150
Before removing exact duplicates: 10150
After removing exact duplicates: 10000
Exact duplicates remaining: 0
Duplicate Order_IDs: 0

Category distribution:
Category
Electronic        711
ELECTRONICS       693
Electronis        676
electronics       668
Fashion           663
FASHION           638
Electronics       629
Fasion            622
fashion           619
Clothing          586
home & kitchen    382
Home              381
Home & Kitchen    368
H&K               368
beauty            367
BEAUTY            351
Personal Care     348
HOME & KITCHEN    327
Beuty             306
Beauty            297
Name: count, dtype: int64


In [5]:
# Clean and validate Order_ID

# Preserve missing values, then strip whitespace
orders['Order_ID'] = orders['Order_ID'].astype('string').str.strip()

# Check missing IDs
print("Missing Order_IDs:", orders['Order_ID'].isna().sum())

# Validate KartZone Order_ID format
valid_order_id = orders['Order_ID'].str.match(
    r'^ORD-KZ-\d+$',
    na=False
)

print("Invalid Order_IDs:", (~valid_order_id).sum())

# Remove rows with missing or invalid Order_ID
orders = orders[valid_order_id].copy()
orders = orders.reset_index(drop=True)

print("Rows after Order_ID validation:", len(orders))
print("Duplicate Order_IDs:", orders['Order_ID'].duplicated().sum())

Missing Order_IDs: 0
Invalid Order_IDs: 0
Rows after Order_ID validation: 10000
Duplicate Order_IDs: 0


In [6]:
# Validate Customer_ID and Product_ID
# Flag invalid foreign keys, but do NOT remove orders yet

# Clean whitespace first
orders['Customer_ID'] = orders['Customer_ID'].astype('string').str.strip()
orders['Product_ID'] = orders['Product_ID'].astype('string').str.strip()

# Load valid IDs from final cleaned tables
valid_customers = pd.read_csv(
    'KartZone_Customers_Final.csv'
)['Customer_ID'].astype('string').str.strip()

valid_products = pd.read_csv(
    'KartZone_Products_Final.csv'
)['Product_ID'].astype('string').str.strip()

# Check foreign key validity
orders['Is_Valid_Customer'] = orders['Customer_ID'].isin(valid_customers)
orders['Is_Valid_Product'] = orders['Product_ID'].isin(valid_products)

print("Invalid Customer_IDs:",
      (~orders['Is_Valid_Customer']).sum())

print("Invalid Product_IDs:",
      (~orders['Is_Valid_Product']).sum())

print("\nSample invalid customers:")
print(
    orders.loc[
        ~orders['Is_Valid_Customer'],
        ['Order_ID', 'Customer_ID']
    ].head()
)

print("\nSample invalid products:")
print(
    orders.loc[
        ~orders['Is_Valid_Product'],
        ['Order_ID', 'Product_ID']
    ].head()
)

print("\nRows retained for now:", len(orders))

Invalid Customer_IDs: 218
Invalid Product_IDs: 189

Sample invalid customers:
          Order_ID Customer_ID
37   ORD-KZ-100038       C9133
46   ORD-KZ-100047       C9124
48   ORD-KZ-100049       C9689
83   ORD-KZ-100084       C9269
144  ORD-KZ-100145       C9957

Sample invalid products:
          Order_ID    Product_ID
88   ORD-KZ-100089  INVALID-7674
215  ORD-KZ-100216  INVALID-9927
275  ORD-KZ-100276  INVALID-4576
295  ORD-KZ-100296  INVALID-8822
301  ORD-KZ-100302  INVALID-7692

Rows retained for now: 10000


In [7]:
# ============================================================
# STEP 5 — PARSE AND VALIDATE DATE COLUMNS
# ============================================================

def parse_date(val):
    if pd.isnull(val) or str(val).strip() in [
        '', 'NA', 'N/A', 'nan', 'null', '-'
    ]:
        return np.nan

    val = str(val).strip()

    formats = [
        '%Y-%m-%d',
        '%d/%m/%Y',
        '%d-%m-%Y',
        '%m/%d/%Y',
        '%d.%m.%Y',
        '%Y/%m/%d'
    ]

    for fmt in formats:
        try:
            return datetime.strptime(val, fmt).strftime('%Y-%m-%d')
        except:
            continue

    return np.nan


date_cols = ['Order_Date', 'Expected_Delivery_Date']

for col in date_cols:
    orders[col] = orders[col].apply(parse_date)
    orders[col] = pd.to_datetime(
        orders[col],
        errors='coerce'
    )

# Order_Date is critical
print("Null Order_Date:",
      orders['Order_Date'].isnull().sum())

# Remove orders with no valid Order_Date
orders = orders.dropna(
    subset=['Order_Date']
).reset_index(drop=True)

print("\nDate column data types:")
print(orders[date_cols].dtypes)

print("\nMissing dates:")
print(orders[date_cols].isnull().sum())

Null Order_Date: 0

Date column data types:
Order_Date                datetime64[ns]
Expected_Delivery_Date    datetime64[ns]
dtype: object

Missing dates:
Order_Date                   0
Expected_Delivery_Date    1826
dtype: int64


In [8]:
# Standardize Order_Status

def standardize_status(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower()
    if val in ['delivered','deliverd','complete','completed']:
        return 'Delivered'
    if val in ['returned','return','refunded']:
        return 'Returned'
    if val in ['cancelled','canceled','cancel']:
        return 'Cancelled'
    if val in ['pending','in transit','processing','shipped']:
        return 'Pending'
    return np.nan

orders['Order_Status'] = orders['Order_Status'].apply(standardize_status)

# Fill nulls with mode
orders['Order_Status'] = orders['Order_Status'].fillna(
    orders['Order_Status'].mode()[0]
)

print(orders['Order_Status'].value_counts())

Order_Status
Delivered    6482
Returned     1427
Cancelled    1283
Pending       808
Name: count, dtype: int64


In [9]:
#Standardize Payment_Mode
def standardize_payment(val):
    if pd.isnull(val):
        return 'Unknown'
    val = str(val).strip().lower()
    if 'upi' in val:                          return 'UPI'
    if 'credit' in val or val == 'cc':        return 'Credit Card'
    if 'debit' in val or val == 'dc':         return 'Debit Card'
    if 'cod' in val or 'cash' in val:         return 'COD'
    if 'net' in val or val == 'nb':           return 'Net Banking'
    if 'emi' in val:                          return 'EMI'
    if 'wallet' in val or 'paytm' in val or 'phonepe' in val: return 'Wallet'
    return 'Other'

orders['Payment_Mode'] = orders['Payment_Mode'].apply(standardize_payment)

print(orders['Payment_Mode'].value_counts())

Payment_Mode
UPI            3445
COD            1805
Credit Card    1515
Debit Card     1176
Net Banking     812
EMI             736
Wallet          413
Other            98
Name: count, dtype: int64


In [10]:
#Standardize Category

def standardize_category(val):
    if pd.isnull(val):
        return np.nan
    val = str(val).strip().lower()
    if 'electron' in val:                    return 'Electronics'
    if 'fashion' in val or 'cloth' in val:   return 'Fashion'
    if 'home' in val or 'kitchen' in val:    return 'Home & Kitchen'
    if 'beauty' in val or 'personal' in val: return 'Beauty'
    return np.nan

orders['Category'] = orders['Category'].apply(standardize_category)

# Derive from Product_ID prefix if still null
def derive_cat(pid):
    if pd.isnull(pid): return np.nan
    pid = str(pid).upper()
    if pid.startswith('ELEC'): return 'Electronics'
    if pid.startswith('FASH'): return 'Fashion'
    if pid.startswith('HOME'): return 'Home & Kitchen'
    if pid.startswith('BEAU'): return 'Beauty'
    return np.nan

orders['Category'] = orders.apply(
    lambda row: derive_cat(row['Product_ID'])
    if pd.isnull(row['Category']) else row['Category'],
    axis=1
)

print(orders['Category'].value_counts())


Category
Electronics       3377
Fashion           3113
Home & Kitchen    1815
Beauty            1665
Name: count, dtype: int64


In [11]:
# ============================================================
# STANDARDIZE CITY
# ============================================================

def standardize_city(val):
    if pd.isnull(val):
        return 'Unknown'

    val = str(val).strip().lower()

    city_map = {
        'mumbai': 'Mumbai',
        'bombay': 'Mumbai',

        'delhi': 'Delhi',
        'new delhi': 'Delhi',

        'bangalore': 'Bangalore',
        'bengaluru': 'Bangalore',
        'banglore': 'Bangalore',

        'chennai': 'Chennai',
        'madras': 'Chennai',

        'hyderabad': 'Hyderabad',
        'hyd': 'Hyderabad',
        'hydrabad': 'Hyderabad',

        'pune': 'Pune',
        'poona': 'Pune'
    }

    for key, standard in city_map.items():
        if key in val:
            return standard

    return 'Unknown'


orders['City'] = orders['City'].apply(standardize_city)

print("\nCity distribution after standardization:")
print(orders['City'].value_counts(dropna=False))

print("\nMissing City values:",
      orders['City'].isna().sum())


City distribution after standardization:
City
Mumbai       2530
Delhi        2199
Bangalore    2029
Chennai      1306
Hyderabad    1147
Pune          789
Name: count, dtype: int64

Missing City values: 0


In [12]:
# ============================================================
# STEP — CLEAN QUANTITY AND PRICING COLUMNS
# ============================================================

# 1. Clean Quantity
orders['Quantity'] = pd.to_numeric(
    orders['Quantity'],
    errors='coerce'
)

print(
    "Invalid quantity (<1):",
    (orders['Quantity'] < 1).sum()
)

print(
    "Invalid quantity (>10):",
    (orders['Quantity'] > 10).sum()
)

# Keep only valid quantities between 1 and 10
orders['Quantity'] = orders['Quantity'].where(
    orders['Quantity'].between(1, 10),
    np.nan
)

# Fill missing/invalid quantities with 1
orders['Quantity'] = (
    orders['Quantity']
    .fillna(1)
    .round()
    .astype(int)
)

print("\nQuantity distribution:")
print(
    orders['Quantity']
    .value_counts()
    .sort_index()
)


# ============================================================
# 2. Clean MRP and Pricing Columns
# ============================================================

def clean_price(val):

    if pd.isnull(val):
        return np.nan

    val = str(val).strip()

    # Remove currency symbols and labels
    val = (
        val.replace('₹', '')
           .replace('Rs.', '')
           .replace('Rs', '')
           .replace('INR', '')
           .replace(',', '')
           .strip()
    )

    try:
        result = float(val)

        # Prices must be positive
        if result <= 0:
            return np.nan

        return result

    except (ValueError, TypeError):
        return np.nan


price_cols = [
    'MRP',
    'Selling_Price',
    'Final_Amount',
    'Unit_Cost',
    'Total_Cost'
]

for col in price_cols:
    orders[col] = orders[col].apply(clean_price)


print("\nPricing summary after cleaning:")
print(
    orders[price_cols].describe()
)


# ============================================================
# 3. Recalculate Final_Amount
# ============================================================

# Final_Amount = Selling_Price × Quantity

invalid_final = (
    orders['Final_Amount'].isna()
    | (orders['Final_Amount'] <= 0)
)

print(
    "\nInvalid / missing Final_Amount before recalculation:",
    invalid_final.sum()
)

orders.loc[invalid_final, 'Final_Amount'] = (
    orders.loc[invalid_final, 'Selling_Price']
    * orders.loc[invalid_final, 'Quantity']
).round(2)


# ============================================================
# 4. Recalculate Total_Cost
# ============================================================

# Total_Cost = Unit_Cost × Quantity

orders['Total_Cost'] = (
    orders['Unit_Cost']
    * orders['Quantity']
).round(2)


# ============================================================
# 5. Final Validation
# ============================================================

print("\n" + "=" * 60)
print("  NUMERIC / PRICING CLEANING VALIDATION")
print("=" * 60)

print(
    "Null Quantity:",
    orders['Quantity'].isnull().sum()
)

print(
    "Null MRP:",
    orders['MRP'].isnull().sum()
)

print(
    "Null Selling_Price:",
    orders['Selling_Price'].isnull().sum()
)

print(
    "Null Final_Amount:",
    orders['Final_Amount'].isnull().sum()
)

print(
    "Null Unit_Cost:",
    orders['Unit_Cost'].isnull().sum()
)

print(
    "Null Total_Cost:",
    orders['Total_Cost'].isnull().sum()
)

print(
    "Invalid Final_Amount (<= 0):",
    (orders['Final_Amount'] <= 0).sum()
)

print("\nFinal_Amount summary:")
print(
    orders['Final_Amount'].describe()
)

print("\nTotal_Cost summary:")
print(
    orders['Total_Cost'].describe()
)

print("\nNumeric and pricing cleaning complete.")

Invalid quantity (<1): 0
Invalid quantity (>10): 0

Quantity distribution:
Quantity
1    5431
2    2327
3    1061
4     719
5     462
Name: count, dtype: int64

Pricing summary after cleaning:
                 MRP  Selling_Price   Final_Amount     Unit_Cost  \
count   10000.000000   10000.000000    8462.000000  10000.000000   
mean    19953.774816   16775.245173   32646.296188  13666.394573   
std     27213.804340   22968.079920   56456.749897  19719.254960   
min       119.460000     101.540000     103.350000     80.740000   
25%      2736.727500    2371.435000    3463.522500   1410.445000   
50%      6675.690000    5582.730000    8889.945000   3415.880000   
75%     26713.977500   22112.537500   35248.425000  18878.182500   
max    111449.080000  107425.440000  513024.000000  69970.430000   

          Total_Cost  
count   10000.000000  
mean    26446.295363  
std     47882.162728  
min        80.740000  
25%      2111.252500  
50%      5466.470000  
75%     28317.762500  
max    349

In [14]:
# ============================================================
# CLEAN DISCOUNT_PCT
# ============================================================

def clean_discount(val):
    if pd.isnull(val):
        return np.nan

    val = str(val).strip().replace('%', '')

    try:
        d = float(val)

        # Valid business range: 0% to 70%
        if 0 <= d <= 70:
            return d

        return np.nan

    except (ValueError, TypeError):
        return np.nan


# Clean Discount_Pct
orders['Discount_Pct'] = orders['Discount_Pct'].apply(clean_discount)


# ============================================================
# FILL MISSING DISCOUNTS
# ============================================================

# Use category median to preserve category-level pricing patterns
orders['Discount_Pct'] = (
    orders.groupby('Category')['Discount_Pct']
    .transform(lambda x: x.fillna(x.median()))
)

# Safety fallback in case a category has no valid values
orders['Discount_Pct'] = orders['Discount_Pct'].fillna(
    orders['Discount_Pct'].median()
)


# ============================================================
# FINAL VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("  DISCOUNT CLEANING VALIDATION")
print("=" * 60)

print(
    "Remaining null Discount_Pct:",
    orders['Discount_Pct'].isnull().sum()
)

print(
    "Invalid Discount_Pct (<0):",
    (orders['Discount_Pct'] < 0).sum()
)

print(
    "Invalid Discount_Pct (>70):",
    (orders['Discount_Pct'] > 70).sum()
)

print("\nDiscount_Pct summary:")
print(
    orders['Discount_Pct'].describe()
)


# ============================================================
# DISCOUNT BY CATEGORY
# ============================================================

print("\nDiscount by Category:")
print(
    orders.groupby('Category')['Discount_Pct']
    .agg(['count', 'mean', 'min', 'max'])
    .round(2)
)


# ============================================================
# FINAL SAFETY CHECK
# ============================================================

assert orders['Discount_Pct'].isnull().sum() == 0
assert (orders['Discount_Pct'] < 0).sum() == 0
assert (orders['Discount_Pct'] > 70).sum() == 0

print("\nDiscount cleaning complete.")
print("All Discount_Pct values are between 0% and 70%.")


  DISCOUNT CLEANING VALIDATION
Remaining null Discount_Pct: 0
Invalid Discount_Pct (<0): 0
Invalid Discount_Pct (>70): 0

Discount_Pct summary:
count    10000.000000
mean        15.263200
std         11.941524
min          0.000000
25%          5.000000
50%         15.000000
75%         20.000000
max         70.000000
Name: Discount_Pct, dtype: float64

Discount by Category:
                count   mean  min   max
Category                               
Beauty           1665  12.29  0.0  70.0
Electronics      3377  17.26  0.0  69.0
Fashion          3113  18.74  0.0  70.0
Home & Kitchen   1815   8.31  0.0  67.0

Discount cleaning complete.
All Discount_Pct values are between 0% and 70%.


In [15]:
# ============================================================
# STEP — PROFIT, RETURN REASON & COUPON CLEANING
# ============================================================

# ------------------------------------------------------------
# 1. RECALCULATE PROFIT
# ------------------------------------------------------------
# Profit = Final_Amount - Total_Cost

orders['Profit'] = (
    orders['Final_Amount'] - orders['Total_Cost']
).round(2)


# ------------------------------------------------------------
# 2. RECALCULATE PROFIT MARGIN
# ------------------------------------------------------------
# Profit Margin % = Profit / Final_Amount * 100

orders['Profit_Margin_Pct'] = np.where(
    orders['Final_Amount'] > 0,
    (orders['Profit'] / orders['Final_Amount']) * 100,
    np.nan
)

orders['Profit_Margin_Pct'] = (
    orders['Profit_Margin_Pct']
    .round(2)
)


# ------------------------------------------------------------
# 3. VALIDATE PROFIT & PROFIT MARGIN
# ------------------------------------------------------------

print("=" * 60)
print("  PROFIT CLEANING VALIDATION")
print("=" * 60)

print("\nProfit nulls:",
      orders['Profit'].isnull().sum())

print("Profit Margin nulls:",
      orders['Profit_Margin_Pct'].isnull().sum())

print("\nProfit summary:")
print(orders['Profit'].describe())

print("\nProfit Margin summary:")
print(orders['Profit_Margin_Pct'].describe())


# ------------------------------------------------------------
# 4. CLEAN RETURN_REASON
# ------------------------------------------------------------

# Convert common missing-value strings to actual NaN
orders['Return_Reason'] = orders['Return_Reason'].replace(
    [
        'nan', 'NaN', 'NA', 'N/A',
        'None', '', 'null', '-'
    ],
    np.nan
)


# Check invalid return reasons
non_returned_with_reason = (
    (orders['Order_Status'] != 'Returned') &
    (orders['Return_Reason'].notna())
).sum()

print(
    f"\nNon-returned orders with return reason: "
    f"{non_returned_with_reason}"
)


# Remove return reasons from non-returned orders
orders.loc[
    orders['Order_Status'] != 'Returned',
    'Return_Reason'
] = np.nan


# Fill missing reason for returned orders
returned_missing_reason = (
    (orders['Order_Status'] == 'Returned') &
    (orders['Return_Reason'].isna())
)

print(
    "Returned orders missing reason:",
    returned_missing_reason.sum()
)

orders.loc[
    returned_missing_reason,
    'Return_Reason'
] = 'Reason Not Provided'


# Display return reason distribution
print("\nReturn Reason Distribution:")
print(
    orders['Return_Reason']
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 5. CLEAN COUPON_CODE
# ------------------------------------------------------------

# Standardize missing-value representations
orders['Coupon_Code'] = orders['Coupon_Code'].replace(
    [
        'nan', 'NaN', 'NA', 'N/A',
        'None', '', 'null', '-'
    ],
    np.nan
)


# Remove unnecessary whitespace
orders['Coupon_Code'] = (
    orders['Coupon_Code']
    .astype('string')
    .str.strip()
)


# Convert coupon codes to uppercase
orders['Coupon_Code'] = (
    orders['Coupon_Code']
    .str.upper()
)


# ------------------------------------------------------------
# 6. VALIDATE COUPONS
# ------------------------------------------------------------

print("\nCoupon Distribution:")
print(
    orders['Coupon_Code']
    .value_counts(dropna=False)
    .head(15)
)

print(
    "\nOrders with coupon:",
    orders['Coupon_Code'].notna().sum()
)

print(
    "Orders without coupon:",
    orders['Coupon_Code'].isna().sum()
)


# ------------------------------------------------------------
# 7. FINAL VALIDATION FOR THIS STEP
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("  PROFIT / RETURN / COUPON CLEANING COMPLETE")
print("=" * 60)

print("\nRemaining nulls:")
print(
    orders[
        ['Profit', 'Profit_Margin_Pct',
         'Return_Reason', 'Coupon_Code']
    ].isnull().sum()
)

  PROFIT CLEANING VALIDATION

Profit nulls: 0
Profit Margin nulls: 0

Profit summary:
count     10000.000000
mean       6957.568784
std       17485.137654
min      -51455.100000
25%         852.105000
50%        2496.070000
75%        6779.850000
max      439162.840000
Name: Profit, dtype: float64

Profit Margin summary:
count    10000.000000
mean        31.815949
std         19.828460
min        -37.220000
25%         18.210000
50%         32.170000
75%         46.240000
max         92.730000
Name: Profit_Margin_Pct, dtype: float64

Non-returned orders with return reason: 0
Returned orders missing reason: 326

Return Reason Distribution:
Return_Reason
NaN                        8573
Reason Not Provided         326
Not as Described            138
Product Defective           123
Product Damaged             115
Changed Mind                114
Better Price Available      109
Duplicate Order             108
Size/Fit Issue              101
Wrong Item Delivered        100
Quality Not as Expe

In [16]:
# ============================================================
# CLEAN IS_FIRST_ORDER
# ============================================================

def clean_bool(val):
    if pd.isnull(val):
        return np.nan

    val = str(val).strip().lower()

    if val in ['yes', '1', 'true', 'y']:
        return 1

    if val in ['no', '0', 'false', 'n']:
        return 0

    return np.nan


orders['Is_First_Order'] = orders['Is_First_Order'].apply(clean_bool)

# Missing values treated as "No"
orders['Is_First_Order'] = (
    orders['Is_First_Order']
    .fillna(0)
    .astype(int)
)

print("\nIs_First_Order distribution:")
print(orders['Is_First_Order'].value_counts())

print(
    "\nMissing Is_First_Order:",
    orders['Is_First_Order'].isnull().sum()
)


Is_First_Order distribution:
Is_First_Order
0    7498
1    2502
Name: count, dtype: int64

Missing Is_First_Order: 0


In [17]:
# ============================================================
# OUTLIER DETECTION — DO NOT CAP BUSINESS VALUES
# ============================================================
# We identify statistical outliers for analysis, but we do not
# modify the original business values.
#
# Reason:
#   Final_Amount, Profit, Discount_Pct and Profit_Margin_Pct
#   may contain legitimate high/low business values.
#   Blind IQR capping would distort revenue and profitability.

def detect_iqr_outliers(series, col_name):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((series < lower) | (series > upper)).sum()

    print(
        f"  {col_name}: {outliers} statistical outliers "
        f"| bounds [{lower:.2f}, {upper:.2f}]"
    )

    return outliers, lower, upper


print("\nOutlier detection:")

outlier_summary = {}

for col in [
    'Final_Amount',
    'Discount_Pct',
    'Profit',
    'Profit_Margin_Pct'
]:
    outlier_summary[col] = detect_iqr_outliers(
        orders[col],
        col
    )

print("\nOutlier detection complete.")
print("No business values were capped or modified.")


Outlier detection:
  Final_Amount: 1129 statistical outliers | bounds [-43294.06, 81296.54]
  Discount_Pct: 361 statistical outliers | bounds [-17.50, 42.50]
  Profit: 1166 statistical outliers | bounds [-8039.51, 15671.47]
  Profit_Margin_Pct: 48 statistical outliers | bounds [-23.84, 88.28]

Outlier detection complete.
No business values were capped or modified.


In [18]:
# ============================================================
# STEP — FOREIGN KEY VALIDATION
# ============================================================

print("=" * 60)
print("FOREIGN KEY VALIDATION")
print("=" * 60)

# Load cleaned master tables
customers_master = pd.read_csv(
    'KartZone_Customers_Final.csv'
)

products_master = pd.read_csv(
    'KartZone_Products_Final.csv'
)

# Clean master IDs
valid_customers = (
    customers_master['Customer_ID']
    .astype('string')
    .str.strip()
)

valid_products = (
    products_master['Product_ID']
    .astype('string')
    .str.strip()
)

# Clean order IDs
orders['Customer_ID'] = (
    orders['Customer_ID']
    .astype('string')
    .str.strip()
)

orders['Product_ID'] = (
    orders['Product_ID']
    .astype('string')
    .str.strip()
)

# Validate Customer_ID
orders['Is_Valid_Customer'] = (
    orders['Customer_ID'].isin(valid_customers)
)

# Validate Product_ID
orders['Is_Valid_Product'] = (
    orders['Product_ID'].isin(valid_products)
)

print(
    "Invalid Customer_IDs:",
    (~orders['Is_Valid_Customer']).sum()
)

print(
    "Invalid Product_IDs:",
    (~orders['Is_Valid_Product']).sum()
)

# Show examples before removal
print("\nSample invalid customers:")
print(
    orders.loc[
        ~orders['Is_Valid_Customer'],
        ['Order_ID', 'Customer_ID']
    ].head()
)

print("\nSample invalid products:")
print(
    orders.loc[
        ~orders['Is_Valid_Product'],
        ['Order_ID', 'Product_ID']
    ].head()
)

# Remove orphan orders
invalid_fk = (
    (~orders['Is_Valid_Customer']) |
    (~orders['Is_Valid_Product'])
)

print(
    "\nOrders removed due to invalid foreign keys:",
    invalid_fk.sum()
)

orders = orders.loc[~invalid_fk].copy()
orders = orders.reset_index(drop=True)

# Remove temporary validation columns
orders = orders.drop(
    columns=['Is_Valid_Customer', 'Is_Valid_Product']
)

print(
    "Rows after foreign-key validation:",
    len(orders)
)

print(
    "Remaining invalid Customer_IDs:",
    (~orders['Customer_ID'].isin(valid_customers)).sum()
)

print(
    "Remaining invalid Product_IDs:",
    (~orders['Product_ID'].isin(valid_products)).sum()
)

print("=" * 60)
print("FOREIGN KEY VALIDATION COMPLETE")
print("=" * 60)

FOREIGN KEY VALIDATION
Invalid Customer_IDs: 218
Invalid Product_IDs: 189

Sample invalid customers:
          Order_ID Customer_ID
37   ORD-KZ-100038       C9133
46   ORD-KZ-100047       C9124
48   ORD-KZ-100049       C9689
83   ORD-KZ-100084       C9269
144  ORD-KZ-100145       C9957

Sample invalid products:
          Order_ID    Product_ID
88   ORD-KZ-100089  INVALID-7674
215  ORD-KZ-100216  INVALID-9927
275  ORD-KZ-100276  INVALID-4576
295  ORD-KZ-100296  INVALID-8822
301  ORD-KZ-100302  INVALID-7692

Orders removed due to invalid foreign keys: 402
Rows after foreign-key validation: 9598
Remaining invalid Customer_IDs: 0
Remaining invalid Product_IDs: 0
FOREIGN KEY VALIDATION COMPLETE


In [19]:
#Feature Engineering

today = pd.Timestamp.now()

# 1. Date-based features
orders['Order_Year']    = orders['Order_Date'].dt.year
orders['Order_Month']   = orders['Order_Date'].dt.month
orders['Order_Quarter'] = orders['Order_Date'].dt.quarter
orders['Order_DayOfWeek'] = orders['Order_Date'].dt.day_name()
orders['Is_Weekend']    = orders['Order_Date'].dt.dayofweek.isin([5,6]).astype(int)

# 2. Festive Season Flag — Oct Nov Dec
orders['Is_Festive_Season'] = orders['Order_Month'].isin([10,11,12]).astype(int)

# 3. Order Status Flags
orders['Is_Delivered']  = (orders['Order_Status'] == 'Delivered').astype(int)
orders['Is_Returned']   = (orders['Order_Status'] == 'Returned').astype(int)
orders['Is_Cancelled']  = (orders['Order_Status'] == 'Cancelled').astype(int)

# 4. High Discount Flag
orders['High_Discount_Flag'] = (orders['Discount_Pct'] > 20).astype(int)

# 5. Coupon Used Flag
orders['Coupon_Used'] = orders['Coupon_Code'].notna().astype(int)

# 6. Is Loss Order
orders['Is_Loss_Order'] = (orders['Profit'] < 0).astype(int)
print(f"Loss-making orders: {orders['Is_Loss_Order'].sum()}")

# 7. Order Value Band
def order_value_band(amt):
    if pd.isnull(amt): return 'Unknown'
    if amt < 500:        return 'Low (< 500)'
    elif amt < 2000:     return 'Medium (500-2K)'
    elif amt < 10000:    return 'High (2K-10K)'
    else:                return 'Premium (10K+)'

orders['Order_Value_Band'] = orders['Final_Amount'].apply(order_value_band)

# 8. COD Flag
orders['Is_COD'] = (orders['Payment_Mode'] == 'COD').astype(int)

# 9. Days to Expected Delivery
orders['Expected_Delivery_Days'] = (
    orders['Expected_Delivery_Date'] - orders['Order_Date']
).dt.days

# 10. Revenue Contribution Percent
total_revenue = orders['Final_Amount'].sum()
orders['Revenue_Contribution_Pct'] = round(
    orders['Final_Amount'] / total_revenue * 100, 4
)

print(orders[['Is_Festive_Season','Is_Returned','High_Discount_Flag',
              'Is_Loss_Order','Order_Value_Band','Is_COD']].head(10))

# Normalization
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Min-Max Normalization
scaler_mm = MinMaxScaler()
orders['Final_Amount_Normalized'] = scaler_mm.fit_transform(
    orders[['Final_Amount']]
)
orders['Discount_Normalized'] = scaler_mm.fit_transform(
    orders[['Discount_Pct']]
)

# Z-Score Standardization
scaler_z = StandardScaler()
orders['Profit_Zscore'] = scaler_z.fit_transform(
    orders[['Profit']]
)

# Log Transformation — Final_Amount is right skewed
orders['Final_Amount_Log'] = np.log1p(orders['Final_Amount'])
orders['Profit_Log']       = np.log1p(orders['Profit'].clip(lower=0))

print(orders[['Final_Amount_Normalized','Discount_Normalized',
              'Profit_Zscore','Final_Amount_Log']].describe())

#Encoding Categorical Variables

# 1. One Hot Encoding — Category
cat_dummies = pd.get_dummies(
    orders['Category'], prefix='Cat', drop_first=False
)
orders = pd.concat([orders, cat_dummies], axis=1)

# 2. One Hot Encoding — Payment_Mode
pay_dummies = pd.get_dummies(
    orders['Payment_Mode'], prefix='Pay', drop_first=True
)
orders = pd.concat([orders, pay_dummies], axis=1)

# 3. Label Encoding — Order_Status
status_order = {'Cancelled':0,'Pending':1,'Returned':2,'Delivered':3}
orders['Status_Encoded'] = orders['Order_Status'].map(status_order)

# 4. Month Encoding — Cyclical (important for seasonality)
orders['Month_Sin'] = np.sin(2 * np.pi * orders['Order_Month'] / 12)
orders['Month_Cos'] = np.cos(2 * np.pi * orders['Order_Month'] / 12)

# 5. Day of Week Encoding — Cyclical
orders['DayOfWeek_Num'] = orders['Order_Date'].dt.dayofweek
orders['Day_Sin'] = np.sin(2 * np.pi * orders['DayOfWeek_Num'] / 7)
orders['Day_Cos'] = np.cos(2 * np.pi * orders['DayOfWeek_Num'] / 7)

print(f"Shape after encoding: {orders.shape}")
print(orders[['Status_Encoded','Month_Sin','Month_Cos']].head())

Loss-making orders: 544
   Is_Festive_Season  Is_Returned  High_Discount_Flag  Is_Loss_Order  \
0                  1            1                   1              0   
1                  0            0                   0              0   
2                  0            1                   0              0   
3                  0            0                   0              0   
4                  1            0                   0              0   
5                  1            0                   0              0   
6                  0            0                   1              0   
7                  1            0                   0              0   
8                  0            0                   1              0   
9                  0            0                   0              0   

  Order_Value_Band  Is_COD  
0  Medium (500-2K)       0  
1  Medium (500-2K)       1  
2  Medium (500-2K)       0  
3  Medium (500-2K)       0  
4   Premium (10K+)       0  
5    High

In [20]:
#Final Validation

print("\n" + "="*55)
print("  ORDERS — FINAL DATA QUALITY REPORT")
print("="*55)
print(f"  Total rows          : {len(orders):,}")
print(f"  Total columns       : {len(orders.columns)}")
print(f"  Remaining nulls     : {orders.isnull().sum().sum():,}")
print(f"  Duplicates          : {orders.duplicated().sum()}")

print(f"\n  BUSINESS METRICS:")
total_rev  = orders['Final_Amount'].sum()
total_prof = orders['Profit'].sum()
margin     = total_prof / total_rev * 100
print(f"  Total Revenue       : ₹{total_rev:,.2f}")
print(f"  Total Profit        : ₹{total_prof:,.2f}")
print(f"  Overall Margin      : {margin:.2f}%")
print(f"  Loss-making orders  : {orders['Is_Loss_Order'].sum():,}")
print(f"  Return rate         : {orders['Is_Returned'].mean()*100:.2f}%")
print(f"  Cancellation rate   : {orders['Is_Cancelled'].mean()*100:.2f}%")
print(f"  COD orders          : {orders['Is_COD'].mean()*100:.2f}%")
print(f"  Festive orders      : {orders['Is_Festive_Season'].mean()*100:.2f}%")

print(f"\n  ORDER STATUS:")
print(orders['Order_Status'].value_counts())

print(f"\n  CATEGORY REVENUE:")
print(orders.groupby('Category')['Final_Amount'].sum().sort_values(ascending=False))

print(f"\n  MONTHLY TREND:")
print(orders.groupby('Order_Month')['Final_Amount'].sum().round(0))

# Save full cleaned version
orders.to_csv('KartZone_Orders_Clean.csv', index=False)
print("Saved → KartZone_Orders_Clean.csv ✓")

# Essential columns for SQL load
essential_cols = [
    'Order_ID','Customer_ID','Product_ID','Order_Date',
    'Expected_Delivery_Date','Category','City','Payment_Mode',
    'Quantity','MRP','Discount_Pct','Selling_Price',
    'Final_Amount','Unit_Cost','Total_Cost','Profit',
    'Profit_Margin_Pct','Order_Status','Return_Reason',
    'Coupon_Code','Is_First_Order',
    'Order_Year','Order_Month','Order_Quarter',
    'Is_Weekend','Is_Festive_Season',
    'Is_Delivered','Is_Returned','Is_Cancelled',
    'High_Discount_Flag','Coupon_Used','Is_Loss_Order',
    'Order_Value_Band','Is_COD','Revenue_Contribution_Pct'
]
orders[essential_cols].to_csv('KartZone_Orders_Final.csv', index=False)
print("Saved → KartZone_Orders_Final.csv ✓")


  ORDERS — FINAL DATA QUALITY REPORT
  Total rows          : 9,598
  Total columns       : 59
  Remaining nulls     : 14,472
  Duplicates          : 0

  BUSINESS METRICS:
  Total Revenue       : ₹311,154,406.30
  Total Profit        : ₹67,027,158.72
  Overall Margin      : 21.54%
  Loss-making orders  : 544
  Return rate         : 14.27%
  Cancellation rate   : 12.91%
  COD orders          : 18.02%
  Festive orders      : 43.51%

  ORDER STATUS:
Order_Status
Delivered    6220
Returned     1370
Cancelled    1239
Pending       769
Name: count, dtype: int64

  CATEGORY REVENUE:
Category
Electronics       2.634577e+08
Fashion           2.395976e+07
Home & Kitchen    1.882839e+07
Beauty            4.908593e+06
Name: Final_Amount, dtype: float64

  MONTHLY TREND:
Order_Month
1     20720582.0
2     19506306.0
3     19447130.0
4     18292835.0
5     18617322.0
6     15910932.0
7     19503778.0
8     19501137.0
9     22160647.0
10    43046552.0
11    55149523.0
12    39297662.0
Name: Final_Am

In [21]:
# ============================================================
# FINAL ORDERS SANITY CHECK
# ============================================================

print("\n" + "="*60)
print("  FINAL ORDERS SANITY CHECK")
print("="*60)

# 1. Primary key
print("\n1. ORDER_ID CHECK")
print("Null Order_IDs:", orders['Order_ID'].isna().sum())
print("Duplicate Order_IDs:", orders['Order_ID'].duplicated().sum())

# 2. Foreign keys
print("\n2. FOREIGN KEY CHECK")

valid_customers = (
    pd.read_csv('KartZone_Customers_Final.csv')['Customer_ID']
    .astype('string')
    .str.strip()
)

valid_products = (
    pd.read_csv('KartZone_Products_Final.csv')['Product_ID']
    .astype('string')
    .str.strip()
)

print(
    "Invalid Customer_IDs:",
    (~orders['Customer_ID'].isin(valid_customers)).sum()
)

print(
    "Invalid Product_IDs:",
    (~orders['Product_ID'].isin(valid_products)).sum()
)

# 3. Numeric validity
print("\n3. NUMERIC CHECK")

print("Invalid Quantity:",
      ((orders['Quantity'] < 1) | (orders['Quantity'] > 10)).sum())

print("Invalid MRP:",
      (orders['MRP'] <= 0).sum())

print("Invalid Selling_Price:",
      (orders['Selling_Price'] <= 0).sum())

print("Invalid Final_Amount:",
      (orders['Final_Amount'] <= 0).sum())

print("Invalid Unit_Cost:",
      (orders['Unit_Cost'] <= 0).sum())

print("Invalid Total_Cost:",
      (orders['Total_Cost'] <= 0).sum())

print("Invalid Discount:",
      ((orders['Discount_Pct'] < 0) |
       (orders['Discount_Pct'] > 70)).sum())

# 4. Business logic
print("\n4. BUSINESS LOGIC CHECK")

print(
    "Profit calculation mismatches:",
    (
        orders['Profit'].round(2) !=
        (orders['Final_Amount'] - orders['Total_Cost']).round(2)
    ).sum()
)

print(
    "Profit margin mismatches:",
    (
        orders['Profit_Margin_Pct'].round(2) !=
        (
            orders['Profit'] /
            orders['Final_Amount'] *
            100
        ).round(2)
    ).sum()
)

print(
    "Non-returned orders with Return_Reason:",
    (
        (orders['Order_Status'] != 'Returned') &
        orders['Return_Reason'].notna()
    ).sum()
)

print(
    "Returned orders without Return_Reason:",
    (
        (orders['Order_Status'] == 'Returned') &
        orders['Return_Reason'].isna()
    ).sum()
)

# 5. Category consistency with Product master
print("\n5. CATEGORY CONSISTENCY CHECK")

product_master = pd.read_csv('KartZone_Products_Final.csv')

product_master['Product_ID'] = (
    product_master['Product_ID']
    .astype('string')
    .str.strip()
)

product_master['Category'] = (
    product_master['Category']
    .astype('string')
    .str.strip()
)

orders_check = orders[
    ['Order_ID', 'Product_ID', 'Category']
].merge(
    product_master[['Product_ID', 'Category']],
    on='Product_ID',
    how='left',
    suffixes=('_Order', '_Product')
)

category_mismatch = (
    orders_check['Category_Order'] !=
    orders_check['Category_Product']
)

print(
    "Product/Order category mismatches:",
    category_mismatch.sum()
)

# 6. Final shape
print("\n6. FINAL SHAPE")
print("Rows:", len(orders))
print("Columns:", len(orders.columns))

print("\n" + "="*60)
print("  SANITY CHECK COMPLETE")
print("="*60)


  FINAL ORDERS SANITY CHECK

1. ORDER_ID CHECK
Null Order_IDs: 0
Duplicate Order_IDs: 0

2. FOREIGN KEY CHECK
Invalid Customer_IDs: 0
Invalid Product_IDs: 0

3. NUMERIC CHECK
Invalid Quantity: 0
Invalid MRP: 0
Invalid Selling_Price: 0
Invalid Final_Amount: 0
Invalid Unit_Cost: 0
Invalid Total_Cost: 0
Invalid Discount: 0

4. BUSINESS LOGIC CHECK
Profit calculation mismatches: 0
Profit margin mismatches: 0
Non-returned orders with Return_Reason: 0
Returned orders without Return_Reason: 0

5. CATEGORY CONSISTENCY CHECK
Product/Order category mismatches: 0

6. FINAL SHAPE
Rows: 9598
Columns: 59

  SANITY CHECK COMPLETE
